# T25 - 政府网站链接爬取 重构

## 项目概述

本项目是一个政府网站链接爬取系统，用于自动化采集政府网站的公开数据。主要功能包括：

1. **网站发现与识别** - 自动发现和识别政府网站
2. **页面爬取与分析** - 爬取页面内容并进行智能分析
3. **内容分类与存储** - 对爬取的内容进行分类存储
4. **特征学习与优化** - 基于机器学习的特征学习与优化

### 技术架构

- **爬虫引擎**: Playwright + aiohttp (异步爬取)
- **数据库**: MySQL (aiomysql + SQLAlchemy)
- **NLP处理**: jieba分词 + TF-IDF
- **机器学习**: scikit-learn (RandomForest)
- **配置管理**: YAML + 环境变量

---

## 1. 环境配置

导入必要的库并设置环境。

In [ ]:
# 标准库导入
import os
import sys
import asyncio
import re
import json
import logging
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Set, Any, Optional, Tuple
from urllib.parse import urljoin, urlparse
from dataclasses import dataclass, field

# 第三方库导入
import yaml
import aiohttp
import aiomysql
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession
from sqlalchemy.orm import sessionmaker
from sqlalchemy import select, delete

# NLP和机器学习
import jieba
import jieba.analyse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('gov_crawler')

print("环境配置完成!")

---

## 2. 配置管理

加载和管理项目配置。

In [ ]:
class ConfigManager:
    """配置管理器"""
    
    def __init__(self, config_dir: str = './data'):
        self.config_dir = Path(config_dir)
        self._config_cache = {}
        
    def get(self, key: str, default: Any = None) -> Any:
        """获取配置值"""
        keys = key.split('.')
        value = self._config_cache
        for k in keys:
            if isinstance(value, dict) and k in value:
                value = value[k]
            else:
                return default
        return value
    
    def load_config(self, config_name: str) -> Dict:
        """加载YAML配置文件"""
        config_path = self.config_dir / f"{config_name}.yml"
        if not config_path.exists():
            config_path = self.config_dir / f"{config_name}.yaml"
        
        if config_path.exists():
            with open(config_path, 'r', encoding='utf-8') as f:
                config = yaml.safe_load(f)
                self._config_cache[config_name] = config
                return config
        return {}

# 初始化配置管理器
config_manager = ConfigManager()

# 加载爬虫配置
crawler_config = config_manager.load_config('crawler')
print(f"爬虫配置加载完成: {list(crawler_config.keys()) if crawler_config else '使用默认配置'}")

# 加载数据库配置
db_config = config_manager.load_config('database')
print(f"数据库配置加载完成: {list(db_config.keys()) if db_config else '使用默认配置'}")

In [ ]:
# 项目配置常量
class Config:
    """项目配置"""
    
    # 数据库配置
    DB_HOST = os.environ.get('DB_HOST', 'localhost')
    DB_PORT = int(os.environ.get('DB_PORT', 3306))
    DB_NAME = os.environ.get('DB_NAME', 'site_dis')
    DB_USER = os.environ.get('DB_USER', 'root')
    DB_PASSWORD = os.environ.get('DB_PASSWORD', '')
    
    # 爬虫配置
    MAX_CONCURRENT_PAGES = 5
    MAX_CONCURRENT_SITES = 3
    PAGE_TIMEOUT = 30000  # 毫秒
    MAX_RETRIES = 3
    RETRY_DELAY = 5  # 秒
    
    # 请求限制
    MAX_REQUESTS_PER_SECOND = 10
    MAX_REQUESTS_PER_MINUTE = 600
    
    # 缓存配置
    CACHE_ENABLED = True
    CACHE_EXPIRE = 3600  # 秒
    
    # 代理配置
    PROXY_ENABLED = os.environ.get('PROXY_ENABLED', 'false').lower() == 'true'
    PROXY_URL = os.environ.get('PROXY_URL', '')

print(f"配置加载完成 - DB: {Config.DB_HOST}:{Config.DB_PORT}/{Config.DB_NAME}")

---

## 3. 数据获取

实现网站数据采集的核心功能。

In [ ]:
# 枚举定义
from enum import Enum

class SiteType(Enum):
    """网站类型"""
    GOVERNMENT = 'government'
    NEWS = 'news'
    GENERAL = 'general'
    UNKNOWN = 'unknown'

class SiteStatus(Enum):
    """网站状态"""
    ACTIVE = 'active'
    INACTIVE = 'inactive'
    PENDING = 'pending'
    ERROR = 'error'

class PageType(Enum):
    """页面类型"""
    GOV_INFO = 'gov_info'      # 政府信息
    NOTICE = 'notice'          # 公告通知
    NEWS = 'news'              # 新闻动态
    GUIDE = 'guide'            # 办事指南
    UNKNOWN = 'unknown'

class CrawlStatus(Enum):
    """爬取状态"""
    PENDING = 'pending'
    RUNNING = 'running'
    SUCCESS = 'success'
    FAILED = 'failed'

print("枚举定义完成!")

In [ ]:
# 省份配置
PROVINCE_SITES = {
    '北京': 'http://www.beijing.gov.cn',
    '上海': 'http://www.sh.gov.cn',
    '广东': 'http://www.gd.gov.cn',
    '江苏': 'http://www.js.gov.cn',
    '浙江': 'http://www.zj.gov.cn',
    '山东': 'http://www.sd.gov.cn',
    '四川': 'http://www.sc.gov.cn',
    '湖北': 'http://www.hubei.gov.cn',
    '河南': 'http://www.henan.gov.cn',
    '福建': 'http://www.fujian.gov.cn',
}

# 中央部委配置
CENTRAL_GOVERNMENT = [
    {'name': '国家发改委', 'domain': 'ndrc.gov.cn'},
    {'name': '工信部', 'domain': 'miit.gov.cn'},
    {'name': '商务部', 'domain': 'mofcom.gov.cn'},
    {'name': '财政部', 'domain': 'mof.gov.cn'},
    {'name': '人民银行', 'domain': 'pbc.gov.cn'},
]

print(f"已配置 {len(PROVINCE_SITES)} 个省份门户网站")
print(f"已配置 {len(CENTRAL_GOVERNMENT)} 个中央部委网站")

In [ ]:
class URLHelper:
    """URL处理工具类"""
    
    @staticmethod
    def normalize_url(url: str) -> str:
        """规范化URL"""
        if not url:
            return ''
        url = url.strip()
        if not url.startswith(('http://', 'https://')):
            url = 'http://' + url
        # 移除尾部斜杠和锚点
        url = url.rstrip('/')
        if '#' in url:
            url = url.split('#')[0]
        return url
    
    @staticmethod
    def get_domain(url: str) -> str:
        """提取域名"""
        try:
            parsed = urlparse(url)
            return parsed.netloc
        except:
            return ''
    
    @staticmethod
    def is_valid_url(url: str) -> bool:
        """检查URL是否有效"""
        if not url:
            return False
        try:
            result = urlparse(url)
            return all([result.scheme, result.netloc])
        except:
            return False
    
    @staticmethod
    def is_government_domain(domain: str) -> bool:
        """检查是否为政府域名"""
        if not domain:
            return False
        return domain.endswith('.gov.cn')

print("URLHelper 工具类定义完成!")

---

## 4. 数据处理

数据库管理和数据处理功能。

In [ ]:
class DatabaseManager:
    """数据库管理器"""
    
    def __init__(self):
        self.pool = None
        self.engine = None
        self._session_factory = None
        
    async def initialize(self):
        """初始化数据库连接"""
        try:
            # 创建连接池
            self.pool = await aiomysql.create_pool(
                host=Config.DB_HOST,
                port=Config.DB_PORT,
                user=Config.DB_USER,
                password=Config.DB_PASSWORD,
                db=Config.DB_NAME,
                charset='utf8mb4',
                autocommit=True
            )
            logger.info("数据库连接池创建成功")
            
            # 创建异步引擎
            database_url = (
                f"mysql+aiomysql://{Config.DB_USER}:{Config.DB_PASSWORD}@"
                f"{Config.DB_HOST}:{Config.DB_PORT}/{Config.DB_NAME}"
            )
            self.engine = create_async_engine(
                database_url,
                pool_pre_ping=True
            )
            
            # 创建会话工厂
            self._session_factory = sessionmaker(
                self.engine,
                class_=AsyncSession,
                expire_on_commit=False
            )
            
            logger.info("数据库管理器初始化完成")
            
        except Exception as e:
            logger.error(f"数据库初始化失败: {str(e)}")
            raise
    
    async def cleanup(self):
        """清理资源"""
        if self.engine:
            await self.engine.dispose()
        if self.pool:
            self.pool.close()
            await self.pool.wait_closed()
            
    async def execute(self, query: str, params: tuple = None) -> bool:
        """执行SQL语句"""
        async with self.pool.acquire() as conn:
            async with conn.cursor() as cur:
                try:
                    await cur.execute(query, params or ())
                    return True
                except Exception as e:
                    logger.error(f"执行SQL失败: {str(e)}")
                    return False
    
    async def fetch_one(self, query: str, params: tuple = None) -> Optional[Dict]:
        """查询单条记录"""
        async with self.pool.acquire() as conn:
            async with conn.cursor(aiomysql.DictCursor) as cur:
                try:
                    await cur.execute(query, params or ())
                    return await cur.fetchone()
                except Exception as e:
                    logger.error(f"查询失败: {str(e)}")
                    return None
    
    async def fetch_all(self, query: str, params: tuple = None) -> List[Dict]:
        """查询多条记录"""
        async with self.pool.acquire() as conn:
            async with conn.cursor(aiomysql.DictCursor) as cur:
                try:
                    await cur.execute(query, params or ())
                    return await cur.fetchall()
                except Exception as e:
                    logger.error(f"查询失败: {str(e)}")
                    return []

print("DatabaseManager 定义完成!")

In [ ]:
class WebsiteManager:
    """网站管理器"""
    
    def __init__(self, db_manager: DatabaseManager = None):
        self.db_manager = db_manager
        self.province_sites = PROVINCE_SITES
        self._url_cache = {}
        self._feature_cache = {
            'url_patterns': {},
            'domain_patterns': {},
            'section_patterns': {},
            'last_update': datetime.now()
        }
        
    async def initialize(self):
        """初始化管理器"""
        if self.db_manager:
            await self._load_features()
            
    async def _load_features(self):
        """加载特征"""
        # 从数据库加载特征配置
        pass
        
    def get_section_type(self, url: str) -> Optional[str]:
        """根据URL获取栏目类型"""
        url = URLHelper.normalize_url(url)
        
        # 栏目模式匹配
        section_patterns = {
            PageType.GOV_INFO.value: [r'/zwgk/', r'/xxgk/', r'/govinfo/', r'/zfxxgk/'],
            PageType.NOTICE.value: [r'/notice/', r'/gonggao/', r'/tzgg/'],
            PageType.NEWS.value: [r'/news/', r'/xinwen/', r'/dongtai/'],
            PageType.GUIDE.value: [r'/guide/', r'/service/', r'/bszn/'],
        }
        
        for section_type, patterns in section_patterns.items():
            for pattern in patterns:
                if re.search(pattern, url, re.I):
                    return section_type
        return None
    
    def get_province(self, url: str) -> Optional[str]:
        """获取URL所属省份"""
        url = URLHelper.normalize_url(url)
        for province, site_url in self.province_sites.items():
            if url.startswith(site_url):
                return province
        return None
    
    def is_valid_url(self, url: str) -> bool:
        """检查URL是否有效"""
        url = URLHelper.normalize_url(url)
        if not URLHelper.is_valid_url(url):
            return False
        domain = URLHelper.get_domain(url)
        return URLHelper.is_government_domain(domain)

print("WebsiteManager 定义完成!")

---

## 5. 核心逻辑

爬虫核心功能实现。

In [ ]:
class PageAnalyzer:
    """页面分析器"""
    
    # 页面类型特征定义
    PAGE_TYPE_FEATURES = {
        PageType.GOV_INFO.value: {
            'title_keywords': ['关于', '通知', '意见', '办法', '规定', '条例', '实施'],
            'content_keywords': ['各省', '各部门', '文号', '特此', '执行'],
            'url_patterns': [r'/zwgk/', r'/xxgk/', r'/govinfo/'],
        },
        PageType.NOTICE.value: {
            'title_keywords': ['公告', '通告', '通知', '公示', '声明'],
            'content_keywords': ['现将', '特此公告', '公示期', '联系电话'],
            'url_patterns': [r'/notice/', r'/gonggao/', r'/tzgg/'],
        },
        PageType.NEWS.value: {
            'title_keywords': ['新闻', '动态', '资讯', '快讯', '要闻'],
            'content_keywords': ['记者', '报道', '采访', '表示'],
            'url_patterns': [r'/news/', r'/xinwen/', r'/dongtai/'],
        },
    }
    
    def __init__(self):
        # 初始化jieba分词
        self._init_jieba()
        self._stopwords = set()
        
    def _init_jieba(self):
        """初始化jieba分词"""
        # 添加政府相关词汇
        for page_type, features in self.PAGE_TYPE_FEATURES.items():
            for keyword in features['title_keywords'] + features['content_keywords']:
                jieba.add_word(keyword)
                
    def analyze_page(self, html_content: str, url: str = '') -> Dict[str, Any]:
        """分析页面内容"""
        try:
            soup = BeautifulSoup(html_content, 'html.parser')
            
            # 提取基本信息
            title = self._extract_title(soup)
            text = self._extract_text(soup)
            
            # 检测页面类型
            page_type = self._detect_page_type(title, text, url)
            
            # 提取关键词
            keywords = self._extract_keywords(text)
            
            # 提取链接
            links = self._extract_links(soup, url)
            
            return {
                'title': title,
                'text_length': len(text),
                'page_type': page_type,
                'keywords': keywords,
                'link_count': len(links),
                'links': links[:100],  # 限制链接数量
                'analyzed_at': datetime.now().isoformat()
            }
            
        except Exception as e:
            logger.error(f"分析页面失败: {str(e)}")
            return {}
    
    def _extract_title(self, soup: BeautifulSoup) -> str:
        """提取标题"""
        for selector in ['h1', 'title', '.title', '.article-title']:
            element = soup.select_one(selector)
            if element:
                return element.get_text(strip=True)
        return ''
    
    def _extract_text(self, soup: BeautifulSoup) -> str:
        """提取文本内容"""
        # 移除不需要的标签
        for tag in soup.find_all(['script', 'style', 'noscript', 'iframe']):
            tag.decompose()
        return soup.get_text(strip=True)
    
    def _detect_page_type(self, title: str, content: str, url: str) -> str:
        """检测页面类型"""
        scores = {}
        
        for page_type, features in self.PAGE_TYPE_FEATURES.items():
            score = 0
            
            # 标题关键词匹配
            for kw in features['title_keywords']:
                if kw in title:
                    score += 2
            
            # 内容关键词匹配
            for kw in features['content_keywords']:
                if kw in content:
                    score += 1
            
            # URL模式匹配
            for pattern in features['url_patterns']:
                if re.search(pattern, url, re.I):
                    score += 3
            
            scores[page_type] = score
        
        if scores:
            best_type = max(scores.items(), key=lambda x: x[1])
            if best_type[1] > 0:
                return best_type[0]
        
        return PageType.UNKNOWN.value
    
    def _extract_keywords(self, text: str, top_k: int = 10) -> List[str]:
        """提取关键词"""
        try:
            return jieba.analyse.extract_tags(text, topK=top_k)
        except:
            return []
    
    def _extract_links(self, soup: BeautifulSoup, base_url: str) -> List[str]:
        """提取链接"""
        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if href and not href.startswith(('#', 'javascript:', 'mailto:')):
                abs_url = urljoin(base_url, href)
                if URLHelper.is_valid_url(abs_url):
                    links.append(URLHelper.normalize_url(abs_url))
        return list(set(links))

print("PageAnalyzer 定义完成!")

In [ ]:
class AsyncCrawler:
    """异步爬虫"""
    
    def __init__(self, max_concurrent: int = 5):
        self.max_concurrent = max_concurrent
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.session = None
        self.analyzer = PageAnalyzer()
        self._processed_urls = set()
        self._error_urls = set()
        
    async def initialize(self):
        """初始化爬虫"""
        timeout = aiohttp.ClientTimeout(total=30)
        self.session = aiohttp.ClientSession(
            timeout=timeout,
            headers={
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
        )
        logger.info(f"爬虫初始化完成，最大并发数: {self.max_concurrent}")
        
    async def cleanup(self):
        """清理资源"""
        if self.session:
            await self.session.close()
            
    async def fetch_page(self, url: str) -> Optional[str]:
        """获取页面内容"""
        async with self.semaphore:
            try:
                async with self.session.get(url) as response:
                    if response.status == 200:
                        return await response.text()
                    else:
                        logger.warning(f"请求失败 [{url}]: 状态码 {response.status}")
                        return None
            except asyncio.TimeoutError:
                logger.error(f"请求超时 [{url}]")
                return None
            except Exception as e:
                logger.error(f"请求异常 [{url}]: {str(e)}")
                return None
                
    async def crawl_page(self, url: str) -> Dict[str, Any]:
        """爬取单个页面"""
        if url in self._processed_urls:
            return {'status': 'skipped', 'url': url}
            
        self._processed_urls.add(url)
        
        try:
            html = await self.fetch_page(url)
            if html:
                analysis = self.analyzer.analyze_page(html, url)
                return {
                    'status': 'success',
                    'url': url,
                    **analysis
                }
            else:
                self._error_urls.add(url)
                return {'status': 'failed', 'url': url, 'error': '无法获取内容'}
                
        except Exception as e:
            self._error_urls.add(url)
            return {'status': 'failed', 'url': url, 'error': str(e)}
            
    async def crawl_batch(self, urls: List[str]) -> List[Dict[str, Any]]:
        """批量爬取页面"""
        tasks = [self.crawl_page(url) for url in urls]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        processed_results = []
        for i, result in enumerate(results):
            if isinstance(result, Exception):
                processed_results.append({
                    'status': 'error',
                    'url': urls[i],
                    'error': str(result)
                })
            else:
                processed_results.append(result)
                
        return processed_results
    
    def get_stats(self) -> Dict[str, int]:
        """获取爬取统计"""
        return {
            'processed': len(self._processed_urls),
            'errors': len(self._error_urls)
        }

print("AsyncCrawler 定义完成!")

---

## 6. 执行与测试

测试爬虫功能。

In [ ]:
async def test_crawler():
    """测试爬虫功能"""
    crawler = AsyncCrawler(max_concurrent=3)
    
    try:
        await crawler.initialize()
        
        # 测试URL列表
        test_urls = [
            'http://www.gov.cn/xinwen/',
            'http://www.ndrc.gov.cn/xwdt/',
        ]
        
        print(f"开始爬取 {len(test_urls)} 个测试页面...")
        
        results = await crawler.crawl_batch(test_urls)
        
        # 输出结果
        for result in results:
            print(f"\n--- {result.get('url', 'unknown')} ---")
            print(f"状态: {result.get('status')}")
            if result.get('status') == 'success':
                print(f"标题: {result.get('title', 'N/A')}")
                print(f"页面类型: {result.get('page_type', 'N/A')}")
                print(f"文本长度: {result.get('text_length', 0)}")
                print(f"关键词: {result.get('keywords', [])[:5]}")
                print(f"链接数量: {result.get('link_count', 0)}")
            else:
                print(f"错误: {result.get('error', 'Unknown')}")
        
        # 输出统计
        stats = crawler.get_stats()
        print(f"\n=== 爬取统计 ===")
        print(f"已处理: {stats['processed']}")
        print(f"错误数: {stats['errors']}")
        
    finally:
        await crawler.cleanup()

# 运行测试
# await test_crawler()
print("测试函数定义完成，取消注释 await test_crawler() 以运行测试")

In [ ]:
async def main():
    """主程序入口"""
    print("=== 政府网站链接爬取系统 ===")
    print(f"启动时间: {datetime.now().isoformat()}")
    
    # 初始化爬虫
    crawler = AsyncCrawler(max_concurrent=Config.MAX_CONCURRENT_PAGES)
    
    try:
        await crawler.initialize()
        
        # 获取待爬取的URL列表
        urls_to_crawl = list(PROVINCE_SITES.values())[:3]  # 限制数量用于演示
        print(f"准备爬取 {len(urls_to_crawl)} 个网站")
        
        # 执行爬取
        results = await crawler.crawl_batch(urls_to_crawl)
        
        # 处理结果
        success_count = sum(1 for r in results if r.get('status') == 'success')
        print(f"\n爬取完成: 成功 {success_count}/{len(results)}")
        
        # 输出统计
        stats = crawler.get_stats()
        print(f"统计: 已处理 {stats['processed']}, 错误 {stats['errors']}")
        
    finally:
        await crawler.cleanup()
        
    print(f"\n完成时间: {datetime.now().isoformat()}")

# 运行主程序
# await main()
print("主程序定义完成，取消注释 await main() 以运行")

---

## 7. 工具函数（可复用）

常用工具函数集合。

In [ ]:
def extract_text_from_html(html: str) -> str:
    """从HTML中提取纯文本"""
    soup = BeautifulSoup(html, 'html.parser')
    for tag in soup.find_all(['script', 'style', 'noscript']):
        tag.decompose()
    return soup.get_text(strip=True)

def clean_url(url: str) -> str:
    """清理URL"""
    url = url.strip()
    # 移除锚点
    if '#' in url:
        url = url.split('#')[0]
    # 移除尾部斜杠
    return url.rstrip('/')

def is_chinese_text(text: str, threshold: float = 0.3) -> bool:
    """判断文本是否主要是中文"""
    if not text:
        return False
    chinese_count = sum(1 for c in text if '\u4e00' <= c <= '\u9fff')
    return chinese_count / len(text) >= threshold

def format_datetime(dt: datetime = None, fmt: str = '%Y-%m-%d %H:%M:%S') -> str:
    """格式化日期时间"""
    if dt is None:
        dt = datetime.now()
    return dt.strftime(fmt)

def parse_chinese_date(date_str: str) -> Optional[datetime]:
    """解析中文日期格式"""
    patterns = [
        (r'(\d{4})年(\d{1,2})月(\d{1,2})日', '%Y-%m-%d'),
        (r'(\d{4})-(\d{1,2})-(\d{1,2})', '%Y-%m-%d'),
        (r'(\d{4})/(\d{1,2})/(\d{1,2})', '%Y-%m-%d'),
    ]
    for pattern, _ in patterns:
        match = re.search(pattern, date_str)
        if match:
            try:
                year, month, day = match.groups()
                return datetime(int(year), int(month), int(day))
            except:
                pass
    return None

def chunk_list(lst: List, chunk_size: int) -> List[List]:
    """将列表分块"""
    return [lst[i:i + chunk_size] for i in range(0, len(lst), chunk_size)]

def merge_dicts(*dicts: Dict) -> Dict:
    """合并多个字典"""
    result = {}
    for d in dicts:
        result.update(d)
    return result

print("工具函数定义完成!")

In [ ]:
# 数据导出函数
import json
import csv

def export_to_json(data: Any, filepath: str, ensure_ascii: bool = False):
    """导出数据到JSON文件"""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=ensure_ascii, indent=2, default=str)
    print(f"数据已导出到: {filepath}")

def export_to_csv(data: List[Dict], filepath: str, fields: List[str] = None):
    """导出数据到CSV文件"""
    if not data:
        print("无数据可导出")
        return
        
    if fields is None:
        fields = list(data[0].keys())
        
    with open(filepath, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(data)
    print(f"数据已导出到: {filepath}")

def load_from_json(filepath: str) -> Any:
    """从JSON文件加载数据"""
    with open(filepath, 'r', encoding='utf-8') as f:
        return json.load(f)

print("数据导出函数定义完成!")

In [ ]:
print("\n" + "="*50)
print("政府网站链接爬取系统 - Notebook 加载完成")
print("="*50)
print("\n使用说明:")
print("1. 运行所有单元格以初始化系统")
print("2. 取消注释 await test_crawler() 运行测试")
print("3. 取消注释 await main() 运行完整爬取")
print("4. 修改 Config 类配置数据库连接")
print("\n注意事项:")
print("- 爬取前请确保网络连接正常")
print("- 遵守网站的 robots.txt 规则")
print("- 合理设置并发数避免对目标服务器造成压力")